# One campaign, step by step

This notebook walks through a single experiment-selection campaign and then shows
why the ranking of strategies flips between datasets.

Everything runs on **oracle replay**: the dataset is fully measured, the engine
hides it, and a value is revealed only when a strategy chooses that candidate.

If you are running this in Colab, uncomment the clone cell below.

In [1]:
# !git clone https://github.com/<your-username>/sdl-strategy-bench.git
# %cd sdl-strategy-bench
# !pip install -q -r requirements.txt

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sdl_engine import adapters, run_campaign, run_many, summarize
from sdl_engine import plotting, stats, recall_matrix

## 1. The candidate table

Every problem is reduced to the same thing: features `X`, a measured value `y`
oriented so that bigger is better, and nothing else.

In [2]:
ds = adapters.load("crossed_barrel")
print(ds)
print(ds.objective)
print(ds.source)
print()
print("X shape:", ds.X.shape, "| features:", ds.feature_names)
print("y: min %.2f, median %.2f, max %.2f" % (ds.y.min(), np.median(ds.y), ds.y.max()))

<Dataset crossed_barrel: 600 candidates, 4 features, objective='maximise toughness'>
maximise toughness
Gongora et al., 'A Bayesian experimental autonomous researcher for mechanical design', Science Advances 6(15):eaaz1708, 2020. Via Olympus (MIT).

X shape: (600, 4) | features: ['n', 'theta', 'r', 't']
y: min 0.43, median 15.65, max 46.71


## 2. One campaign

A budget of 50 out of 600 candidates. The first 8 are a random initial design;
after that, every step refits the surrogate and picks the highest-scoring
unmeasured candidate.

The initial design is **spread at random, not placed where results look good**.
Clustering it near a promising spot is how a campaign gets trapped on a local
peak.

In [3]:
run = run_campaign(ds, strategy="ei", budget=50, n_init=8, seed=0)

print("picked %d candidates out of %d" % (len(run.chosen), len(ds.y)))
print("best found      : %.2f" % run.best_curve[-1])
print("true optimum    : %.2f" % ds.y.max())
print("%% of optimum    : %.1f%%" % (100 * run.best_curve[-1] / ds.y.max()))
print("top 1%% recovered: %.0f%%" % (100 * run.top_k_recall(ds.y, 0.01)))

picked 50 candidates out of 600
best found      : 44.94
true optimum    : 46.71
% of optimum    : 96.2%
top 1% recovered: 50%


### What the curve looks like

Flat stretches are not wasted. A step that returns a poor value still removes a
region from consideration, and that is what lets the next step jump.

In [4]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, len(run.observed) + 1), run.observed, "o", ms=4,
        color="#9AA3AF", label="value of each experiment")
ax.plot(np.arange(1, len(run.best_curve) + 1), run.best_curve, "-", lw=2.2,
        color="#14213D", label="best so far")
ax.axhline(ds.y.max(), ls="--", lw=1, color="#B4531A", label="true optimum")
ax.axvline(run.n_init, ls=":", lw=1.2, color="#6B7280")
ax.annotate("initial design", xy=(run.n_init, 2), xytext=(run.n_init + 1, 2),
            fontsize=9, color="#6B7280")
ax.set_xlabel("experiment"); ax.set_ylabel("toughness")
ax.set_title("A single EI campaign on crossed_barrel (seed 0)", fontsize=11)
ax.legend(fontsize=9, frameon=False, loc="lower right")
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.grid(alpha=0.25, lw=0.6)
plt.show()

## 3. Four strategies, many seeds

One run is dominated by the luck of the initial design, so each strategy is run
across 20 seeds and the spread is reported alongside the mean.

In [5]:
results = []
for strat in ("random", "greedy", "ucb", "ei"):
    results += run_many(ds, strategies=(strat,), seeds=range(20),
                        budget=50, n_init=8)

pd.DataFrame(summarize(results, ds, k_fracs=(0.01, 0.05))).round(3)

,strategy,n_seeds,final_best_mean,final_best_std,top1pct_recall_mean,top1pct_recall_std
0,random,20,40.261,3.952,0.083,0.124
1,greedy,20,41.260,3.594,0.167,0.167
2,ucb,20,44.776,1.150,0.367,0.113
3,ei,20,44.652,1.622,0.425,0.144


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
plotting.plot_curves(results, ds, ax=axes[0])
plotting.plot_recall(results, ds, ax=axes[1], k_frac=0.01)
plt.tight_layout(); plt.show()

### Which of those gaps are real?

The bars differ in height, but that is not yet a result. Every strategy is run on
the same seed list, and for a given seed they all start from the identical initial
design, so the seeds are **matched pairs** and a paired test is the right one.

Read the confidence interval, not the p-value: if it contains zero, the data
cannot say which strategy is better.


In [ ]:
mat = recall_matrix(results, ds, k_frac=0.01)
print(stats.format_pairwise(stats.pairwise_table(mat)))

res = stats.metric_resolution(len(ds.y), 0.01, values=mat["ei"])
print(f"\ntop 1% of {res['n_candidates']} candidates = {res['n_top']} rows, "
      f"so recall can only move in steps of {res['step_pp']:.1f}pp "
      f"({res['possible_values']} possible values, "
      f"{res['distinct_observed']} distinct seen across {res['n_seeds']} seeds).")
print("A gap smaller than one step is below the resolution of the metric.")


On this dataset the two uncertainty-aware strategies win, and they are also
**steadier** — look at the width of the bands, not only the height of the lines.

Both beat greedy by a margin the paired test confirms. UCB against EI is a
different matter: the interval straddles zero, so the honest statement is that
this benchmark cannot separate them, which is what you would expect from two
members of the same family.

But the ranking itself does not transfer.


## 4. Where it flips

On the two categorical datasets, plain greedy beats UCB and EI. The reason is in
the structure of the data, so it is worth checking rather than guessing.

In [7]:
def top_clustering(df, factor_cols, value_col, k_frac=0.01):
    k = max(1, round(len(df) * k_frac))
    top = df.nlargest(k, value_col)
    rows = []
    for c in factor_cols:
        vc = top[c].value_counts()
        rows.append({"factor": c,
                     "levels in dataset": df[c].nunique(),
                     "levels among top": top[c].nunique(),
                     "most common level": f"{vc.iloc[0]}/{k}"})
    return pd.DataFrame(rows)

bh = pd.read_csv("datasets/buchwald_a.csv", header=None,
                 names=["aryl", "additive", "base", "ligand", "yield"])
print("buchwald_a - how concentrated is the true top 1%?")
display(top_clustering(bh, ["aryl", "additive", "base", "ligand"], "yield"))

buchwald_a - how concentrated is the true top 1%?


,factor,levels in dataset,levels among top,most common level
0,aryl,3,2,4/8
1,additive,22,4,3/8
2,base,3,2,7/8
3,ligand,4,3,4/8


In [8]:
rx_ds = adapters.load("redoxmers")
rx = pd.read_csv("datasets/redoxmers.csv", header=None,
                 names=["r1", "r3", "r4", "r5", "lam", "ered", "gsol"])
rx["score"] = rx_ds.y
print("redoxmers - how concentrated is the true top 1%?")
display(top_clustering(rx, ["r1", "r3", "r4", "r5"], "score"))

redoxmers - how concentrated is the true top 1%?


,factor,levels in dataset,levels among top,most common level
0,r1,2,1,14/14
1,r3,8,3,9/14
2,r4,8,4,8/14
3,r5,11,6,6/14


In [9]:
cb = pd.read_csv("datasets/crossed_barrel.csv", header=None,
                 names=["n", "theta", "r", "t", "toughness"])
k = max(1, round(len(cb) * 0.01))
top = cb.nlargest(k, "toughness")
pd.DataFrame({
    "full range": [f"{cb[c].min():g} - {cb[c].max():g}" for c in ["n","theta","r","t"]],
    "range among top 1%": [f"{top[c].min():g} - {top[c].max():g}" for c in ["n","theta","r","t"]],
}, index=["n","theta","r","t"])

,full range,range among top 1%
n,6 - 12,12 - 12
theta,0 - 200,25 - 200
r,1.5 - 2.5,1.5 - 2.4
t,0.7 - 1.4,0.7 - 1.4


**Read the three tables together.**

In `redoxmers` every candidate in the true top 1% shares the same substituent at
site r1, and most share one r3 group. In `buchwald_a` nearly all of the top
candidates share one base. The good candidates sit inside a single family, so
once the surrogate locates that family, exploiting it harvests the rest — and
exploration spends budget elsewhere.

In `crossed_barrel` the top candidates spread along a ridge: `theta` covers most
of its range and so does `t`. There is no single family to exploit, so
exploration pays.

The practical conclusion is not "use EI" or "use greedy". It is that the
exploration setting should follow from the structure of the problem, and that
structure is something you can check before committing a budget.

## 5. Trying another problem

Add one `Dataset` to `sdl_engine/adapters.py` and nothing else changes:

```python
def load_my_problem():
    df = pd.read_csv(DATA_DIR / "my_data.csv")
    return Dataset(
        key="my_problem",
        title="...",
        X=df[["param_a", "param_b"]].to_numpy(float),
        y=df["measured_value"].to_numpy(float),   # bigger must be better
        feature_names=["param_a", "param_b"],
        objective="maximise ...",
        source="citation",
    )
```

See the limitations section of the README before reading too much into any of
these numbers — this is replay on already-measured data, not a closed loop.